In [41]:
import torch
from torch.utils.data import Dataset
import tiktoken
import json
import numpy as np
from utils.config import GPTConfig

torch.manual_seed(1024)

class nanoDataset(Dataset):
    def __init__(self, path):
        self.char_data = self._read_dataset(path)

    def _read_dataset(self, path):
        text_chunks = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())  # 使用 json.loads 解析单行

                # 将文本按 block_size 切分成小块
                start = 0
                while start < len(data):
                    end = start + 128
                    chunk = data[start:end]
                    if chunk:  # 避免空块
                        text_chunks.append(chunk)
                    start = end

        return text_chunks

    def __len__(self):
        return len(self.char_data)

    def __getitem__(self, idx):
        chunk = self.char_data[idx]
        return chunk

In [2]:
# 添加一个50257作为padding，在进行编码的时候不需要考虑 只需要添加即可 ，但是在进行生成得时候 需要对这个进行特殊的处理 因为并不会decode 50257


In [42]:
config = GPTConfig()
train_dataset = nanoDataset(r'../data/mobvoi_seq_monkey_general_open_corpus_1000.json')

In [43]:
len(train_dataset.char_data)

7590

In [5]:
enc = tiktoken.get_encoding('gpt2')

In [6]:
enc.eot_token

50256

In [7]:
import torch
m = torch.tensor(enc.encode('hello what you name ?'))
m

tensor([31373,   644,   345,  1438,  5633])

In [8]:
torch.concat([m,torch.full(([2]),10)],axis=0)

tensor([31373,   644,   345,  1438,  5633,    10,    10])

In [9]:
m.shape

torch.Size([5])

In [10]:
len(m)

5

In [11]:
from typing import List, Tuple


class nanoCollator:
    def __init__(self, tokenizer, config):
        self.tokenizer = tokenizer
        self.config = config

    def _convert_one_piece_tokens(self, piece):  ## 处理成tensor格式

        input_ids = torch.cat([torch.tensor(self.tokenizer.encode(piece)), torch.tensor([self.tokenizer.eot_token])])
        return input_ids

    def __call__(self, batch: List[str]):
        input_ids_list = []
        max_input_len_list = []
        final_labels_list = []
        for piece in batch:
            inputs_ids = self._convert_one_piece_tokens(piece)
            input_ids_list.append(inputs_ids)
            max_input_len_list.append(len(inputs_ids))

        max_length = max(max_input_len_list) + 2
        final_inputs_list = []
        for index, input_ids in enumerate(input_ids_list):
            pad_len = max_length - max_input_len_list[index]
            pad_input_ids = torch.concat([input_ids, torch.full([pad_len], self.config.pad_token_id)], axis=0)
            pad_labels = torch.concat([input_ids, torch.full([pad_len], self.config.IGNORE_INDEX)], axis=0)
            final_inputs_list.append(pad_input_ids[:-1])
            final_labels_list.append(pad_labels[1:])
        final_input_ids = torch.stack(final_inputs_list, axis=0)
        final_labels = torch.stack(final_labels_list, axis=0)
        attention_mask = torch.ones_like(final_input_ids)
        attention_mask[final_input_ids == self.config.pad_token_id] = 0

        return final_input_ids, attention_mask, final_labels

In [12]:
tokenizer = tiktoken.get_encoding('gpt2')
config = GPTConfig()
nano = nanoCollator(tokenizer,config)

In [13]:
train_dataset[0],train_dataset[1]

('在查处虚开增值税专用发票案件中，常常涉及进项留抵税额和税款损失的认定和处理。在计算税款损失时，要不要将进项留抵税额包括在内？',
 '对此，实务中存在意见分歧。')

In [44]:
ll = []
for i in range(len(train_dataset)):
    if len(train_dataset[i])  < 10:
        ll.append(train_dataset[i])

In [45]:
ll

['路上。',
 '的难题。',
 '间时始终清洁双手。',
 '。',
 '到美白作用。',
 '格的监督和查询。',
 'ail.com\n',
 '年。',
 '目以待。',
 'm。',
 '的最新技术。',
 '社会的言行规范。',
 '特殊情况不准请假。',
 '没狗证。',
 '机正呼唤您的加入。',
 '。',
 '佼佼者，值得一看。',
 '制品。',
 '队协作精神。',
 '点击下载吧。',
 '。',
 '一名奖金1万元。',
 '艺术品。',
 '个交易日，。',
 '精神的重要方式。',
 '的后果不堪设想。',
 '二十大胜利召开 。',
 '管理经验。',
 '。',
 '独立性。',
 '。',
 '学习。',
 '那么重要吗？。',
 '作全部都做好。',
 '联合会杯。',
 '的冲击。',
 '。',
 '。',
 '工程的配套设计。',
 '。',
 '质等。',
 '环境做出一份贡献。',
 '。',
 '很有名的一个人。',
 '子的物质占有欲。',
 '肤。',
 '不再。',
 '空间的方法。',
 '抗病能力较强。',
 '还有待进一步开发。',
 '人物的等级。',
 '了坚实基础。',
 '处办公室联系。',
 '境。',
 '的耐磨陶瓷产品。',
 '到康复。',
 '前景。',
 '重要因素作用。',
 '属国际物流港。',
 '算命。',
 '此的桥梁。',
 '积极治疗皮肤病。']

In [14]:
final_input_ids, attention_mask, final_labels = nano((train_dataset[0],train_dataset[1]))
final_labels.shape, final_input_ids.shape,attention_mask.shape

(torch.Size([2, 152]), torch.Size([2, 152]), torch.Size([2, 152]))

In [15]:
final_labels

tensor([[  101,   162,   253,    98, 13783,   226,   164,   247,   248, 28156,
           222,   161,    95,   252,   161,   222,   120,   163,   101,   236,
         10310,   241, 18796,   101, 20998,   239,   163,    98,   101,   162,
            94,   230, 20015,   114, 40792,   171,   120,   234, 30585,   116,
         30585,   116,   162,   114,   231, 20998,   232, 32573,   249,   165,
            94,   117, 45911,   247,   162,   232,   113,   163,   101,   236,
           165,    95,   251,   161,   240,   234,   163,   101,   236,   162,
           105,   122,   162,   235,   253, 13783,   109, 21410,   164,   106,
            97, 22522,   248,   161,   240,   234, 13783,   226, 49426,   228,
         16764, 28839,   101,   164,   106,    94,   163,   106,   245,   163,
           101,   236,   162,   105,   122,   162,   235,   253, 13783,   109,
         33768,   114,   171,   120,   234, 17358,   223, 38834, 17358,   223,
         49546, 32573,   249,   165,    94,   117, 4

In [16]:
final_input_ids

tensor([[28839,   101,   162,   253,    98, 13783,   226,   164,   247,   248,
         28156,   222,   161,    95,   252,   161,   222,   120,   163,   101,
           236, 10310,   241, 18796,   101, 20998,   239,   163,    98,   101,
           162,    94,   230, 20015,   114, 40792,   171,   120,   234, 30585,
           116, 30585,   116,   162,   114,   231, 20998,   232, 32573,   249,
           165,    94,   117, 45911,   247,   162,   232,   113,   163,   101,
           236,   165,    95,   251,   161,   240,   234,   163,   101,   236,
           162,   105,   122,   162,   235,   253, 13783,   109, 21410,   164,
           106,    97, 22522,   248,   161,   240,   234, 13783,   226, 49426,
           228, 16764, 28839,   101,   164,   106,    94,   163,   106,   245,
           163,   101,   236,   162,   105,   122,   162,   235,   253, 13783,
           109, 33768,   114,   171,   120,   234, 17358,   223, 38834, 17358,
           223, 49546, 32573,   249,   165,    94,  

In [17]:
print(attention_mask)

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [18]:
from model import GPT
from config import GPTConfig


config = GPTConfig()
model = GPT(config).to(config.device)
final_input_ids = final_input_ids.to(config.device)
attention_mask = attention_mask.to(config.device)
final_labels = final_labels.to(config.device)
print(model(final_input_ids,targets = final_labels,padding_attention_mask =None))
# gqa = GroupQueryAttention(config)
# x = torch.randn(2,6,8)
# pa = torch.tensor([[1,1,1,1,0,0],[1,1,0,0,0,0]])
# gqa(x,padding_attention_mask=pa)

(tensor([[ 4.0680e-02,  1.0147e-01,  9.1334e-01,  ..., -1.5880e-02,
         -3.2104e-01,  2.6566e-01],
        [ 1.3618e-01, -5.3648e-02,  7.4473e-01,  ..., -1.2740e-01,
         -8.4211e-04, -8.7525e-03],
        [ 4.5559e-02,  7.8552e-02,  6.0603e-01,  ..., -2.5630e-01,
         -1.0518e-01,  1.7518e-02],
        ...,
        [ 3.3191e-01,  1.5139e-01,  2.7614e-01,  ..., -8.5158e-02,
          1.8152e-01,  4.9345e-01],
        [ 1.9341e-01, -3.8606e-02,  2.7175e-01,  ..., -1.1511e-02,
          2.2594e-01,  5.5181e-01],
        [ 1.5477e-01, -4.8732e-02,  1.9375e-01,  ..., -1.5822e-02,
          1.7734e-01,  5.1086e-01]], device='cuda:0', grad_fn=<ViewBackward0>), tensor(10.8506, device='cuda:0', grad_fn=<NllLossBackward0>))


In [48]:
from torch.utils.data import DataLoader

loader = DataLoader(train_dataset,collate_fn=nano,shuffle=True)

In [49]:
l = []
for id,labels,masks in loader:
    print(id.shape)
    l.append(id.shape[1])

torch.Size([1, 266])
torch.Size([1, 290])
torch.Size([1, 258])
torch.Size([1, 283])
torch.Size([1, 278])
torch.Size([1, 274])
torch.Size([1, 220])
torch.Size([1, 281])
torch.Size([1, 271])
torch.Size([1, 264])
torch.Size([1, 276])
torch.Size([1, 142])
torch.Size([1, 277])
torch.Size([1, 289])
torch.Size([1, 269])
torch.Size([1, 284])
torch.Size([1, 264])
torch.Size([1, 286])
torch.Size([1, 276])
torch.Size([1, 268])
torch.Size([1, 275])
torch.Size([1, 272])
torch.Size([1, 250])
torch.Size([1, 264])
torch.Size([1, 284])
torch.Size([1, 244])
torch.Size([1, 282])
torch.Size([1, 272])
torch.Size([1, 259])
torch.Size([1, 266])
torch.Size([1, 256])
torch.Size([1, 276])
torch.Size([1, 284])
torch.Size([1, 241])
torch.Size([1, 276])
torch.Size([1, 267])
torch.Size([1, 249])
torch.Size([1, 284])
torch.Size([1, 284])
torch.Size([1, 279])
torch.Size([1, 222])
torch.Size([1, 271])
torch.Size([1, 256])
torch.Size([1, 291])
torch.Size([1, 256])
torch.Size([1, 275])
torch.Size([1, 30])
torch.Size([1,

In [50]:
max(l)

320

In [33]:
m = sorted(l,reverse=True)

In [35]:
print(m)

[13378, 10905, 7360, 6485, 4894, 4361, 1984, 1707, 1593, 1584, 1571, 1551, 1458, 1406, 1337, 1283, 1278, 1276, 1269, 1218, 1214, 1212, 1198, 1193, 1154, 1144, 1123, 1113, 1108, 1092, 1085, 1077, 1077, 1069, 1062, 1048, 1046, 1045, 1043, 1041, 1038, 1031, 1031, 1030, 1014, 1014, 1013, 1012, 1011, 988, 988, 988, 986, 985, 984, 970, 969, 967, 963, 963, 953, 947, 946, 945, 942, 940, 940, 935, 935, 934, 932, 932, 929, 928, 926, 923, 917, 915, 912, 902, 901, 894, 888, 887, 887, 884, 879, 875, 870, 866, 865, 864, 863, 856, 856, 852, 852, 851, 849, 845, 841, 841, 841, 832, 831, 828, 828, 825, 822, 814, 809, 808, 808, 806, 804, 803, 801, 800, 799, 798, 796, 796, 792, 791, 788, 782, 779, 778, 778, 777, 777, 773, 772, 767, 764, 763, 763, 759, 752, 748, 748, 748, 743, 743, 742, 732, 730, 730, 729, 728, 725, 724, 724, 724, 720, 720, 720, 719, 718, 718, 715, 715, 713, 713, 713, 712, 711, 709, 709, 708, 706, 704, 704, 700, 697, 695, 694, 693, 692, 691, 687, 686, 685, 684, 683, 682, 681, 679, 678, 677